# Kueue-Managed TransformersTrainer

This notebook demonstrates how to run a `TransformersTrainer` TrainJob through **Kueue** on Red Hat OpenShift AI.

## Overview

In this example, we fine-tune the **Qwen 2.5 1.5B Instruct** model on the **Stanford Alpaca** instruction-following dataset. The training runs on 2 GPU nodes with automatic progress tracking enabled, and the TrainJob is routed through Kueue for advanced scheduling (queueing, gang scheduling, and prioritization).

### What You'll Learn

| Feature | Description |
|---------|-------------|
| **Kueue Scheduling** | Route a TrainJob to a LocalQueue using a Kueue queue label |
| **Automatic Progress Tracking** | Monitor training steps, epochs, loss, and ETA in the OpenShift AI Dashboard |
| **PVC-Based Checkpointing** | Save model checkpoints to a shared PersistentVolumeClaim for durability and resume capability |
| **Distributed Training** | Run training across multiple GPU nodes using PyTorch's DistributedDataParallel (DDP) |

### Model Details

**Qwen 2.5 1.5B Instruct** is a compact instruction-tuned language model from the Qwen family:
- **Parameters:** 1.5 billion
- **Context Length:** 32K tokens
- **Languages:** Multilingual with strong English and Chinese support
- **Use Case:** Ideal for instruction-following, chat, and text generation tasks
- **Why this model?** Small enough to train quickly for demonstration, yet powerful enough for real-world tasks

### Dataset Details

We use the **Stanford Alpaca** dataset (`tatsu-lab/alpaca`), a widely-used instruction-following dataset:

| Property | Value |
|----------|-------|
| **Source** | Stanford University |
| **Size** | 52,000 instruction-response pairs (we use 500 samples for this demo) |
| **Format** | Instruction, optional input, and response |
| **Use Case** | Instruction-tuning language models |

### Prerequisites

Before running this notebook, ensure you have:

1. **OpenShift AI Cluster** with Kubeflow Trainer v2 and Kueue enabled
2. **Workbench** running Python 3.12+ with GPU access
3. **Shared PVC** named `shared` with **ReadWriteMany (RWX)** access mode
   - **Recommended Size:** 20Gi (for model weights, dataset, and checkpoints)
   - **Mount Path:** `/opt/app-root/src/shared` in the workbench
   - See the [README](./README.md) for detailed PVC setup instructions

## Setup and Imports

Install the Kubeflow SDK and required packages.

In [14]:
!pip install kubeflow --no-cache-dir --index-url https://console.redhat.com/api/pypi/public-rhai/rhoai/3.3/cuda12.9-ubi9/simple/
!python3 -m pip install datasets transformers accelerate huggingface_hub

Looking in indexes: https://console.redhat.com/api/pypi/public-rhai/rhoai/3.3/cuda12.9-ubi9/simple/


In [17]:
import os

import kubeflow
import torch
from datasets import load_dataset
from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient
from kubeflow.trainer.options import Labels
from kubeflow.trainer.rhai import TransformersTrainer
from kubernetes import client as k8s
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Kubeflow SDK version: {kubeflow.__version__}")
print("✅ All imports successful")

Kubeflow SDK version: v0.2.1+rhai2
✅ All imports successful


## Configuration

Configure authentication, paths, and Kueue settings. TransformersTrainer has **progress tracking enabled by default**.

### Environment Variables

The following environment variables are required for API authentication:
- `OPENSHIFT_API_URL` - Your OpenShift API server URL (e.g., `https://api.cluster.example.com:6443`)
- `NOTEBOOK_USER_TOKEN` - Authentication token for API access

**Note:** These are typically auto-set in OpenShift AI workbenches. If not set, uncomment and fill in the values in the cell below.

### Kueue LocalQueue

Set a LocalQueue name to route the TrainJob through Kueue. If you leave it empty, the job will be submitted without a Kueue label.

In [21]:
# ============================================================================
# AUTHENTICATION CONFIGURATION
# ============================================================================
# These values are typically auto-set in OpenShift AI workbenches.
# If not set, uncomment and fill in your values below:

# api_server = "https://api.your-cluster.example.com:6443"
# token = "sha256~your-token-here"

# Try to get from environment variables first
api_server = "https://api.privateailab.acic.accenture:6443"
token = "sha256~PqwCeEivYYqOt9lL2LNVxL7mC_E4HFwgvZSdnTGq87w"

if not api_server or not token:
    raise RuntimeError(
        "OPENSHIFT_API_URL and NOTEBOOK_USER_TOKEN environment variables are required.\n"
        "Either set them in your environment, or uncomment and fill in the values above."
    )

# Configure Kubernetes client
configuration = k8s.Configuration()
configuration.host = api_server
configuration.verify_ssl = False  # Set to True if using trusted certificates
configuration.api_key = {"authorization": f"Bearer {token}"}

# ============================================================================
# PVC MOUNT PATHS
# ============================================================================
# Workbench (notebook) mount path
#
# NOTEBOOK_SHARED_PATH is where *your workbench* sees the PVC.
# This depends on what PVC you attached when you created the workbench.
# On OpenShift AI, the default mount convention is:
#   /opt/app-root/src/<pvc-name>
#
# If your PVC is not named "shared", set PVC_NAME accordingly.
PVC_NAME = "shared2"
NOTEBOOK_SHARED_PATH = f"/opt/app-root/src/{PVC_NAME}"

# Training pods mount path
#
# SDK_MOUNT_PATH is a fixed path used by the Kubeflow SDK when you set:
#   TransformersTrainer(output_dir="pvc://<pvc-name>/<path>")
# The SDK mounts the PVC at this location inside the training pods.
# (This comes from the SDK constant CHECKPOINT_MOUNT_PATH.)
SDK_MOUNT_PATH = "/mnt/kubeflow-checkpoints"

# Quick sanity check to help users discover the right workbench mount
if not os.path.exists(NOTEBOOK_SHARED_PATH):
    print(
        "⚠️  Expected workbench PVC mount not found at: "
        f"{NOTEBOOK_SHARED_PATH}\n"
        "If your PVC has a different name or mount, update PVC_NAME/NOTEBOOK_SHARED_PATH.\n"
        "Tip: in a workbench, PVCs are typically under /opt/app-root/src/."
    )

# Model Configuration
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Paths for notebook operations only (downloading model/data, reading checkpoints)
# Note: Training pods use SDK mount convention (/mnt/kubeflow-checkpoints/...)
#       which is handled automatically by the pvc:// URI in TransformersTrainer
NOTEBOOK_MODEL_PATH = f"{NOTEBOOK_SHARED_PATH}/models/qwen2.5-1.5b-instruct"
NOTEBOOK_DATA_PATH = f"{NOTEBOOK_SHARED_PATH}/data/alpaca_processed"
NOTEBOOK_CHECKPOINTS_PATH = (
    f"{NOTEBOOK_SHARED_PATH}/checkpoints/kueue-transformers-trainer"
)

# KUEUE CONFIGURATION
# Set a LocalQueue name to route this TrainJob through Kueue.
# Leave empty to submit without a Kueue queue label.
# If you want to use your LocalQueue, set the name here.
KUEUE_QUEUE_NAME = "team-training"  # Replace with your LocalQueue name

print(f"API Server: {api_server}")
print(f"Model: {MODEL_NAME}")
print(f"PVC name: {PVC_NAME}")
print(f"Workbench PVC mount: {NOTEBOOK_SHARED_PATH}")
print(f"Training pod PVC mount (SDK): {SDK_MOUNT_PATH}")
print(f"Notebook Model Path: {NOTEBOOK_MODEL_PATH}")
print(f"Notebook Data Path: {NOTEBOOK_DATA_PATH}")
print(f"Notebook Checkpoints Path: {NOTEBOOK_CHECKPOINTS_PATH}")
print(f"Kueue queue name: {KUEUE_QUEUE_NAME or 'not set'}")

API Server: https://api.privateailab.acic.accenture:6443
Model: Qwen/Qwen2.5-1.5B-Instruct
PVC name: shared2
Workbench PVC mount: /opt/app-root/src/shared2
Training pod PVC mount (SDK): /mnt/kubeflow-checkpoints
Notebook Model Path: /opt/app-root/src/shared2/models/qwen2.5-1.5b-instruct
Notebook Data Path: /opt/app-root/src/shared2/data/alpaca_processed
Notebook Checkpoints Path: /opt/app-root/src/shared2/checkpoints/kueue-transformers-trainer
Kueue queue name: team-training


## Download Model and Dataset to Shared PVC

Before submitting the training job, we pre-download the model and dataset to the shared PVC. This ensures:
- **Offline Training:** Training pods don't need internet access during training
- **Faster Startup:** No download delays when training pods start
- **Consistency:** All nodes use the same model weights and data

In [22]:
# Download model to PVC
if os.path.exists(NOTEBOOK_MODEL_PATH) and os.listdir(NOTEBOOK_MODEL_PATH):
    print(f"✅ Model already exists at {NOTEBOOK_MODEL_PATH}")
else:
    print(f"🔄 Downloading model {MODEL_NAME} to {NOTEBOOK_MODEL_PATH}...")
    os.makedirs(NOTEBOOK_MODEL_PATH, exist_ok=True)

    # Use fast tokenizer for compatibility
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, use_fast=True, trust_remote_code=True
    )
    tokenizer.save_pretrained(NOTEBOOK_MODEL_PATH)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )
    model.save_pretrained(NOTEBOOK_MODEL_PATH, safe_serialization=True)
    print(f"✅ Model saved to {NOTEBOOK_MODEL_PATH}")
    print(f"📁 Files: {os.listdir(NOTEBOOK_MODEL_PATH)}")

🔄 Downloading model Qwen/Qwen2.5-1.5B-Instruct to /opt/app-root/src/shared2/models/qwen2.5-1.5b-instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to /opt/app-root/src/shared2/models/qwen2.5-1.5b-instruct
📁 Files: ['model.safetensors', 'tokenizer_config.json', 'generation_config.json', 'chat_template.jinja', 'config.json', 'tokenizer.json']


In [23]:
# Download and prepare dataset
if os.path.exists(NOTEBOOK_DATA_PATH) and os.listdir(NOTEBOOK_DATA_PATH):
    print(f"✅ Dataset already exists at {NOTEBOOK_DATA_PATH}")
else:
    print("🔄 Downloading and processing Alpaca dataset...")
    os.makedirs(NOTEBOOK_DATA_PATH, exist_ok=True)

    # Load subset of Alpaca dataset
    dataset = load_dataset("tatsu-lab/alpaca", split="train[:500]")

    # Load tokenizer for preprocessing
    tokenizer = AutoTokenizer.from_pretrained(
        NOTEBOOK_MODEL_PATH, use_fast=True, trust_remote_code=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    def format_instruction(example):
        if example.get("input"):
            text = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
        else:
            text = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"
        return {"text": text}

    dataset = dataset.map(format_instruction, remove_columns=dataset.column_names)

    def tokenize_function(examples):
        tokenized = tokenizer(
            examples["text"],
            padding="max_length",
            truncation=True,
            max_length=512,
        )
        tokenized["labels"] = tokenized["input_ids"].copy()
        return tokenized

    tokenized_dataset = dataset.map(
        tokenize_function, batched=True, remove_columns=["text"]
    )
    tokenized_dataset.save_to_disk(NOTEBOOK_DATA_PATH)
    print(f"✅ Dataset saved to {NOTEBOOK_DATA_PATH}")

print("\n✅ Model and dataset ready on PVC!")

🔄 Downloading and processing Alpaca dataset...


Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

✅ Dataset saved to /opt/app-root/src/shared2/data/alpaca_processed

✅ Model and dataset ready on PVC!


## Define the Training Function

The training function runs inside each training pod as a distributed PyTorch process. TransformersTrainer serializes this function and executes it via `torchrun` on each node.

### Training Configuration

| Parameter | Value | Description |
|-----------|-------|-------------|
| `num_train_epochs` | 1 | Complete one pass through the dataset |
| `per_device_train_batch_size` | 2 | Samples per GPU per step |
| `gradient_accumulation_steps` | 4 | Effective batch size = 2 × 4 × 2 nodes = 16 |
| `learning_rate` | 2e-5 | Standard fine-tuning rate |
| `save_steps` | 20 | Checkpoint every 20 steps |
| `bf16` | True | Use bfloat16 mixed precision |

### Key Points

- **Supported Trainers:** Use `transformers.Trainer` or `trl.SFTTrainer` - both are auto-instrumented
- **No Manual Setup:** Progress tracking callback is injected automatically
- **Local Files Only:** Model and data are loaded from the mounted PVC (no network access needed)

In [24]:
def train_func():
    """SFT training function using HuggingFace Trainer.

    TransformersTrainer automatically:
    - Injects KubeflowProgressCallback for real-time metrics
    - Injects checkpoint instrumentation when `TransformersTrainer.output_dir` is set
    - Auto-resumes from the latest checkpoint when available
    """
    import os

    import torch
    from datasets import load_from_disk
    from transformers import (
        AutoModelForCausalLM,
        DataCollatorForLanguageModeling,
        PreTrainedTokenizerFast,
        Trainer,
        TrainingArguments,
    )

    rank = int(os.environ.get("RANK", 0))
    local_rank = int(os.environ.get("LOCAL_RANK", 0))

    # Model/data are on the shared PVC mounted at /mnt/kubeflow-checkpoints via SDK's pvc:// URI
    model_path = "/mnt/kubeflow-checkpoints/models/qwen2.5-1.5b-instruct"
    data_path = "/mnt/kubeflow-checkpoints/data/alpaca_processed"

    print(f"🚀 Starting training on rank {rank}")

    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)
        print(f"🔧 GPU: {torch.cuda.get_device_name(local_rank)}")

    # Load tokenizer directly from tokenizer.json file
    # This bypasses AutoTokenizer's hub validation that fails with local paths
    print(f"📥 Loading tokenizer from: {model_path}")
    tokenizer_file = os.path.join(model_path, "tokenizer.json")
    tokenizer_config_file = os.path.join(model_path, "tokenizer_config.json")

    # Load tokenizer config to get special tokens
    import json

    with open(tokenizer_config_file) as f:
        tokenizer_config = json.load(f)

    tokenizer = PreTrainedTokenizerFast(
        tokenizer_file=tokenizer_file,
        eos_token=tokenizer_config.get("eos_token", "<|endoftext|>"),
        pad_token=tokenizer_config.get("pad_token"),
        bos_token=tokenizer_config.get("bos_token"),
        unk_token=tokenizer_config.get("unk_token"),
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load model directly from local path
    print(f"📥 Loading model from: {model_path}")
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        device_map={"": local_rank},
        local_files_only=True,
        trust_remote_code=True,
    )

    # Load dataset
    print(f"📥 Loading dataset from: {data_path}")
    tokenized_dataset = load_from_disk(data_path)

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    # TransformersTrainer automatically configures:
    # - output_dir: Set from the pvc:// URI (mounted at /mnt/kubeflow-checkpoints)
    # - save_strategy, save_steps, save_total_limit: Set from PeriodicCheckpointConfig
    training_args = TrainingArguments(
        output_dir="/tmp/output",  # Placeholder - SDK overrides this
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        logging_steps=5,
        report_to="none",
        bf16=True,
        ddp_find_unused_parameters=False,
    )

    # Trainer - TransformersTrainer injects:
    # - KubeflowProgressCallback for real-time metrics
    # - Checkpoint instrumentation when output_dir="pvc://..." is set
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=data_collator,
    )

    print(f"💾 Trainer output_dir: {trainer.args.output_dir}")

    # Train - auto-resumes from latest checkpoint if available
    trainer.train()

    # Save final model (only on rank 0)
    if rank == 0:
        final_path = os.path.join(trainer.args.output_dir, "final")
        os.makedirs(final_path, exist_ok=True)
        trainer.save_model(final_path)
        tokenizer.save_pretrained(final_path)
        print(f"✅ Final model saved to {final_path}")

    print(f"✅ Training complete on rank {rank}")


print("✅ Training function defined")

✅ Training function defined


## Create the Trainer Client

Initialize the TrainerClient with authentication configuration.

In [26]:
# Create client with authentication
api_client = k8s.ApiClient(configuration)

backend_config = KubernetesBackendConfig(
    client_configuration=api_client.configuration,
)

client = TrainerClient(backend_config)
print("✅ TrainerClient created")

# Get the torch-distributed runtime
runtime = client.backend.get_runtime("torch-distributed")
print(f"✅ Using runtime: {runtime.name}")

✅ TrainerClient created
✅ Using runtime: torch-distributed


/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.privateailab.acic.accenture'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## Submit the Training Job with TransformersTrainer

Now we create and submit the distributed training job. The `TransformersTrainer` wraps your training function and handles all the distributed training setup.

### Job Configuration

| Parameter | Value | Description |
|-----------|-------|-------------|
| `num_nodes` | 2 | Number of GPU nodes for distributed training |
| `nvidia.com/gpu` | 1 | GPUs per node |
| `cpu` | 4 | CPU cores per node |
| `memory` | 16Gi | Memory per node |
| `enable_jit_checkpoint` | True | Save checkpoint on pod termination (preemption) |

### Kueue Queue Label

Kueue routes TrainJobs based on a LocalQueue label. We attach the label using SDK options:

`kueue.x-k8s.io/queue-name: <LOCAL_QUEUE_NAME>`

If `KUEUE_QUEUE_NAME` is empty, the job is submitted without the label.

### PVC Mounting via `pvc://` URI

We use the SDK's `output_dir="pvc://shared"` parameter which automatically:
- Mounts the PVC at `/mnt/kubeflow-checkpoints` on all training pods
- Handles all Kubernetes volume configuration without manual setup
- Enables JIT checkpointing on pod termination

This ensures all training nodes can access the same model weights, dataset, and write checkpoints to shared storage.

In [33]:
from kubeflow.trainer.rhai.transformers import PeriodicCheckpointConfig

# Configure periodic checkpointing - SDK injects these into TrainingArguments
checkpoint_config = PeriodicCheckpointConfig(
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
)

# Create TransformersTrainer with progress tracking and PVC-based checkpointing
# The output_dir="pvc://shared/checkpoints/kueue-transformers-trainer" tells the SDK to:
# - Mount PVC "shared" at /mnt/kubeflow-checkpoints on all training pods
# - Set TrainingArguments.output_dir to /mnt/kubeflow-checkpoints/checkpoints/kueue-transformers-trainer
# - Enable JIT checkpointing (saves state on SIGTERM)
# - Auto-resume from latest checkpoint on restart
trainer = TransformersTrainer(
    func=train_func,
    num_nodes=2,
    resources_per_node={
        "nvidia.com/gpu": 1,
        "cpu": "4",
        "memory": "16Gi",
    },
    # Make the training function cleaner: set offline mode at the pod level
    env={
        "HF_HUB_OFFLINE": "1",
        "TRANSFORMERS_OFFLINE": "1",
        "NCCL_IB_DISABLE": "1",        # disabilita InfiniBand
        "NCCL_P2P_DISABLE": "1",       # disabilita P2P diretta
        "NCCL_DEBUG": "INFO",          # per vedere più dettagli
    },
    enable_jit_checkpoint=True,  # Save checkpoint on SIGTERM (preemption)
    periodic_checkpoint_config=checkpoint_config,  # Configure periodic saves
    output_dir="pvc://shared2/checkpoints/kueue-transformers-trainer",  # SDK handles mounting
)

kueue_options = []
if KUEUE_QUEUE_NAME:
    kueue_options = [Labels(labels={"kueue.x-k8s.io/queue-name": KUEUE_QUEUE_NAME})]
    print(f"✅ Kueue enabled with LocalQueue: {KUEUE_QUEUE_NAME}")
else:
    print("ℹ️  Kueue queue name not set; submitting without Kueue label")

# Submit the training job - SDK handles PVC mounting automatically
job_name = client.train(
    trainer=trainer,
    runtime=runtime,
    options=kueue_options or None,
)
print(f"✅ Training job submitted: {job_name}")
print("📊 Progress tracking: ENABLED (auto-injected by TransformersTrainer)")
print(f"💾 Checkpoints will appear in workbench at: {NOTEBOOK_CHECKPOINTS_PATH}")

✅ Kueue enabled with LocalQueue: team-training
✅ Training job submitted: g9ee29a5ba0e
📊 Progress tracking: ENABLED (auto-injected by TransformersTrainer)
💾 Checkpoints will appear in workbench at: /opt/app-root/src/shared2/checkpoints/kueue-transformers-trainer


/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.privateailab.acic.accenture'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## Follow Job Logs

Let's fetch our job logs to make sure training is going as expected. The logs will stream in real-time as the training progresses.

In [ ]:
# Stream logs (press Ctrl+C to stop if you want to continue with other cells)
for logline in client.get_job_logs(job_name, follow=True):
    print(logline, end="")

## Get Job Status

Check the final status of the training job after completion.

In [ ]:
# Check job status
job = client.get_job(job_name)
print("Final TrainJob Status:")
print(f"   Name: {job.name}")
print(f"   Status: {job.status}")
print(f"   Created: {job.creation_timestamp}")
print(f"   Nodes: {job.num_nodes}")
print(f"   Runtime: {job.runtime.name}")

if job.steps:
    print("   Steps:")
    for step in job.steps:
        print(f"     - {step.name}: {step.status}")
    print()

## Check Training Progress

View training progress directly in the OpenShift AI Dashboard (Training Jobs).

## Test the Trained Model (Optional)

After training completes, you can load the fine-tuned model from the checkpoint saved on the shared PVC.

### Checkpoint Structure

The training function saves checkpoints with this structure:
```
/opt/app-root/src/shared/checkpoints/kueue-transformers-trainer/
├── checkpoint-<step>/  # Intermediate checkpoints (saved every save_steps)
├── checkpoint-<N>/    # Checkpoint at final step (N = last step number)
└── final/             # Final merged model ready for inference
```

### Testing the Model

We'll load the fine-tuned model and test it with an instruction prompt using the same format as the training data:

```
### Instruction:
<your instruction here>

### Response:
<model generates response>
```

In [ ]:
def find_most_recent_checkpoint(output_dir):
    """Find the most recently created checkpoint directory."""
    if not os.path.exists(output_dir):
        raise FileNotFoundError(f"Output directory not found: {output_dir}")

    checkpoint_dirs = [
        os.path.join(output_dir, d)
        for d in os.listdir(output_dir)
        if os.path.isdir(os.path.join(output_dir, d))
        and (d.startswith("checkpoint-") or d == "final")
    ]

    if not checkpoint_dirs:
        raise FileNotFoundError(f"No checkpoints found in {output_dir}")

    # Prefer 'final' if it exists
    final_path = os.path.join(output_dir, "final")
    if os.path.exists(final_path):
        return final_path

    return max(checkpoint_dirs, key=os.path.getctime)


print("✅ Checkpoint utility defined")

In [ ]:
# Find and load the trained model
final_checkpoint = find_most_recent_checkpoint(NOTEBOOK_CHECKPOINTS_PATH)
print(f"📂 Loading checkpoint from: {final_checkpoint}")

trained_tokenizer = AutoTokenizer.from_pretrained(
    final_checkpoint, trust_remote_code=True
)
trained_model = AutoModelForCausalLM.from_pretrained(
    final_checkpoint,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
    trust_remote_code=True,
)

print("✅ Model loaded successfully")
print(f"📊 Model parameters: {trained_model.num_parameters():,}")

# Test the model
test_prompt = "### Instruction:\nExplain what machine learning is in one sentence.\n\n### Response:"

print("\n📝 Testing model with prompt:")
print(test_prompt)
print("\n🤖 Model response:")

inputs = trained_tokenizer(test_prompt, return_tensors="pt").to(trained_model.device)

# Remove token_type_ids if present (not used by some models like Qwen)
if "token_type_ids" in inputs:
    del inputs["token_type_ids"]

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.95,
    )

response = trained_tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response.replace(test_prompt, "").strip())

print("\n✅ Model test completed!")

## Cleanup

Delete the training job and free resources.

**Note:** To fully clean up, you may also want to delete the downloaded model, dataset, and checkpoints from the PVC:
```bash
rm -rf /opt/app-root/src/shared/models/qwen2.5-1.5b-instruct
rm -rf /opt/app-root/src/shared/data/alpaca_processed
rm -rf /opt/app-root/src/shared/checkpoints/kueue-transformers-trainer
```

In [ ]:
# Delete the training job
client.delete_job(name=job_name)
print(f"✅ Job {job_name} deleted")

In [ ]:
import gc

# Clear CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

gc.collect()
print("✅ Resources freed, CUDA cache cleared")

## Summary

Congratulations! You've successfully completed a Kueue-managed distributed fine-tuning job on OpenShift AI.

### What You Accomplished

| Step | Description |
|------|-------------|
| ✅ Model Download | Downloaded Qwen 2.5 1.5B Instruct to shared PVC |
| ✅ Dataset Preparation | Processed Stanford Alpaca dataset for instruction-tuning |
| ✅ Distributed Training | Ran 2-node distributed training with PyTorch DDP |
| ✅ Kueue Scheduling | Routed the TrainJob to a LocalQueue for Kueue management |
| ✅ Progress Tracking | Monitored real-time metrics via SDK and Dashboard |
| ✅ Checkpointing | Saved model checkpoints to shared PVC |
| ✅ Model Testing | Loaded and tested the fine-tuned model |

### Key Takeaways

1. **TransformersTrainer** automatically instruments your HuggingFace `Trainer` with:
   - `KubeflowProgressCallback` for real-time metrics (HTTP endpoint on port 28080)
   - Progress visible in OpenShift AI Dashboard without any code changes

2. **Kueue routing** is enabled by adding a LocalQueue label via SDK options:
   - `Labels(labels={"kueue.x-k8s.io/queue-name": "<LOCAL_QUEUE_NAME>"})`
   - Leave the queue name empty to submit without Kueue

3. **Shared PVC Strategy** for distributed training:
   - Pre-download model and dataset to shared RWX PVC from workbench
   - Training pods mount the same PVC and access data locally (offline mode)
   - Checkpoints are written to shared storage for durability

4. **Supported Trainers**:
   - `transformers.Trainer` - Standard HuggingFace trainer
   - `trl.SFTTrainer` - TRL's supervised fine-tuning trainer

### How Progress Tracking Works

When you use `TransformersTrainer` with `enable_progression_tracking=True` (the default):

1. **Automatic Instrumentation:** TransformersTrainer injects a `KubeflowProgressCallback` into your HuggingFace `Trainer`
2. **HTTP Metrics Server:** A lightweight HTTP server starts on port 28080, exposing metrics as JSON
3. **Dashboard Integration:** OpenShift AI Dashboard polls these metrics and displays real-time progress

### TransformersTrainer Quick Reference

| Parameter | Description | Default |
|-----------|-------------|---------|
| `func` | Training function using `transformers.Trainer` | Required |
| `num_nodes` | Number of distributed training nodes | Required |
| `resources_per_node` | GPU, CPU, memory per node | Required |
| `enable_progression_tracking` | Enable real-time metrics server | `True` |
| `metrics_poll_interval_seconds` | How often controller polls metrics | `30` |

### Next Steps

- **Scale Up:** Increase `num_nodes` for larger models or datasets
- **Use LoRA:** Add PEFT/LoRA for memory-efficient fine-tuning
- **Try Other Models:** This pattern works with any HuggingFace model
- **Enable JIT Checkpointing:** Use `enable_jit_checkpoint=True` for automatic checkpoint saving on preemption

### Resources

- [Kubeflow Trainer Documentation](https://www.kubeflow.org/docs/components/trainer/)
- [HuggingFace Transformers](https://huggingface.co/docs/transformers/)
- [Stanford Alpaca Dataset](https://huggingface.co/datasets/tatsu-lab/alpaca)